In [ ]:
# -*- coding: utf-8 -*-
"""
Understanding and Implementing a Transformer in PyTorch.

This script provides a detailed, commented implementation of a Transformer model
using PyTorch's built-in `nn.Transformer` module. It's designed to be a learning
resource, explaining each component and parameter of the Transformer architecture.

We will:
1.  Explain the core concepts: Positional Encoding, Multi-Head Self-Attention,
    Encoder, and Decoder.
2.  Build a `nn.Transformer` model, explaining every argument.
3.  Create a simple synthetic dataset for a sequence-to-sequence "sorting" task.
4.  Implement a training loop to train the model.
5.  Run a simple evaluation to see the model in action.
"""

import torch
import torch.nn as nn
import torch.optim as optim
import math
import numpy as np

# Acknowledging the current time and location for context.
# Current time: Tuesday, September 2, 2025 at 6:56 PM IST.
# Location: Kharagpur, West Bengal, India.

# =============================================================================
# 1. ARCHITECTURE COMPONENTS
# =============================================================================

# The Transformer architecture relies on a few key components that are crucial
# to understand before diving into the full model.

class PositionalEncoding(nn.Module):
    """
    Injects some information about the relative or absolute position of the tokens
    in the sequence. The positional encodings have the same dimension as the
    embeddings so that the two can be summed. Here, we use sine and cosine
    functions of different frequencies.
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create a long enough `pe` matrix that can be sliced for any sequence length.
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe) # register_buffer makes it part of the model's state_dict, but not a parameter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor, shape [seq_len, batch_size, embedding_dim]
        """
        # Add positional encoding to the input tensor.
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

# =============================================================================
# 2. THE TRANSFORMER MODEL
# =============================================================================

# Now, let's define the full Transformer model. We'll use PyTorch's `nn.Transformer`.

class Seq2SeqTransformer(nn.Module):
    def __init__(self,
                 num_encoder_layers: int,
                 num_decoder_layers: int,
                 emb_size: int,
                 nhead: int,
                 src_vocab_size: int,
                 tgt_vocab_size: int,
                 dim_feedforward: int = 512,
                 dropout: float = 0.1):
        """
        Initializes the Transformer model.

        Args:
            num_encoder_layers: The number of sub-encoder-layers in the encoder (required).
            num_decoder_layers: The number of sub-decoder-layers in the decoder (required).
            emb_size: The number of expected features in the input (required).
            nhead: The number of heads in the multiheadattention models (required).
            src_vocab_size: The size of the source vocabulary (required).
            tgt_vocab_size: The size of the target vocabulary (required).
            dim_feedforward: The dimension of the feedforward network model (default=512).
            dropout: The dropout value (default=0.1).
        """
        super().__init__()
        self.model_type = 'Transformer'

        # --- Embedding Layers ---
        self.src_tok_emb = nn.Embedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = nn.Embedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(emb_size, dropout=dropout)

        # --- PyTorch's Transformer Module ---
        # The core of our model. It encapsulates the encoder and decoder stacks.
        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=False # IMPORTANT: PyTorch Transformer expects (seq_len, batch, feature)
        )

        # --- Output Layer ---
        # This layer maps the decoder output to the target vocabulary size.
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(self,
                src: torch.Tensor,
                trg: torch.Tensor,
                src_mask: torch.Tensor,
                tgt_mask: torch.Tensor,
                src_padding_mask: torch.Tensor,
                tgt_padding_mask: torch.Tensor,
                memory_key_padding_mask: torch.Tensor):
        """
        Defines the forward pass of the model.
        """
        # 1. Embed the source and target sequences.
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(trg))

        # 2. Pass the embedded sequences and masks to the transformer.
        outs = self.transformer(src_emb, tgt_emb, src_mask, tgt_mask, None,
                                src_padding_mask, tgt_padding_mask, memory_key_padding_mask)

        # 3. Apply the final linear layer.
        return self.generator(outs)

    def encode(self, src: torch.Tensor, src_mask: torch.Tensor):
        return self.transformer.encoder(self.positional_encoding(self.src_tok_emb(src)), src_mask)

    def decode(self, tgt: torch.Tensor, memory: torch.Tensor, tgt_mask: torch.Tensor):
        return self.transformer.decoder(self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask)


# --- Helper Functions for Masks ---
# Masks are crucial in Transformers to prevent the model from "cheating" by
# looking at future tokens or paying attention to padding tokens.

def generate_square_subsequent_mask(sz):
    """
    Generates a square mask for the sequence. The masked positions are filled with -inf.
    Unmasked positions are filled with 0.0. This is used by the decoder to prevent
    attending to future tokens.
    """
    mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

def create_mask(src, tgt, pad_idx, device):
    """
    Creates all the necessary masks for the source and target sequences.
    """
    src_seq_len = src.shape[0]
    tgt_seq_len = tgt.shape[0]

    # Target mask: prevents attending to future tokens.
    tgt_mask = generate_square_subsequent_mask(tgt_seq_len).to(device)
    # Source mask: not typically needed for the encoder, but can be used for specific tasks.
    src_mask = torch.zeros((src_seq_len, src_seq_len), device=device).type(torch.bool)

    # Padding masks: prevent attending to padding tokens.
    src_padding_mask = (src == pad_idx).transpose(0, 1)
    tgt_padding_mask = (tgt == pad_idx).transpose(0, 1)
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask


# =============================================================================
# 3. DATASET AND TRAINING SETUP
# =============================================================================
# We'll create a simple toy task: sorting a sequence of numbers.
# For example, if the input is `[BOS, 3, 1, 4, EOS]`, the target should be
# `[BOS, 1, 3, 4, EOS]`.

# --- Hyperparameters and Constants ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SRC_VOCAB_SIZE = 12  # 0-9 for numbers, 10 for PAD, 11 for BOS/EOS
TGT_VOCAB_SIZE = 12
EMB_SIZE = 512        # d_model: embedding dimension
NHEAD = 8             # Number of attention heads
FFN_HID_DIM = 512     # Feedforward network hidden dimension
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3
DROPOUT = 0.1
PAD_IDX, BOS_IDX, EOS_IDX = 10, 11, 11 # Using the same token for BOS and EOS for simplicity

# --- Model Initialization ---
transformer = Seq2SeqTransformer(NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, EMB_SIZE,
                                 NHEAD, SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, FFN_HID_DIM, DROPOUT)

# Initialize weights for better training performance
for p in transformer.parameters():
    if p.dim() > 1:
        nn.init.xavier_uniform_(p)

transformer = transformer.to(DEVICE)

# --- Loss and Optimizer ---
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

# --- Data Generation ---
def generate_data(batch_size, seq_len):
    """Generates a batch of source and target sequences."""
    for _ in range(batch_size):
        # Generate a random sequence of numbers (0-9)
        seq = np.random.randint(0, 10, size=seq_len)
        src = np.concatenate(([BOS_IDX], seq, [EOS_IDX]))
        # The target is the sorted sequence
        tgt = np.concatenate(([BOS_IDX], np.sort(seq), [EOS_IDX]))
        yield torch.tensor(src), torch.tensor(tgt)

# =============================================================================
# 4. TRAINING AND EVALUATION LOOPS
# =============================================================================

def train_epoch(model, optimizer):
    model.train()
    losses = 0
    # Generate a "training set" of 100 batches
    train_data = list(generate_data(100, seq_len=8))

    for src, tgt in train_data:
        src = src.to(DEVICE).unsqueeze(1) # Add batch dimension
        tgt = tgt.to(DEVICE).unsqueeze(1) # Add batch dimension

        # Prepare target data for the decoder
        # The decoder input should be shifted right (starts with BOS)
        tgt_input = tgt[:-1, :]
        # The ground truth for loss calculation should not include the BOS token
        tgt_out = tgt[1:, :]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input, PAD_IDX, DEVICE)

        # Forward pass
        logits = model(src, tgt_input, src_mask, tgt_mask, src_padding_mask, tgt_padding_mask, src_padding_mask)

        optimizer.zero_grad()

        # Calculate loss
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        loss.backward()

        optimizer.step()
        losses += loss.item()

    return losses / len(train_data)


def evaluate(model):
    model.eval()
    losses = 0
    eval_data = list(generate_data(100, seq_len=8))

    with torch.no_grad():
        for src, tgt in eval_data:
            src = src.to(DEVICE).unsqueeze(1)
            tgt = tgt.to(DEVICE).unsqueeze(1)

            tgt_input = tgt[:-1, :]
            tgt_out = tgt[1:, :]

            src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input, PAD_IDX, DEVICE)

            logits = model(src, tgt_input, src_mask, tgt_mask, src_padding_mask, tgt_padding_mask, src_padding_mask)

            loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
            losses += loss.item()

    return losses / len(eval_data)


# --- Run Training ---
NUM_EPOCHS = 20
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(transformer, optimizer)
    val_loss = evaluate(transformer)
    print(f"Epoch: {epoch}, Train loss: {train_loss:.3f}, Val loss: {val_loss:.3f}")


# =============================================================================
# 5. INFERENCE EXAMPLE
# =============================================================================
# This function shows how to use the trained model to make predictions on new data.
# It uses a greedy approach, selecting the most likely token at each step.

def greedy_decode(model, src, src_mask, max_len, start_symbol):
    model.eval()
    src = src.to(DEVICE)
    src_mask = src_mask.to(DEVICE)

    memory = model.encode(src, src_mask)
    memory = memory.to(DEVICE)
    
    # Start with the beginning-of-sequence token
    ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(DEVICE)
    
    for i in range(max_len - 1):
        tgt_mask = (generate_square_subsequent_mask(ys.size(0))
                    .type(torch.bool)).to(DEVICE)
        
        out = model.decode(ys, memory, tgt_mask)
        out = out.transpose(0, 1)
        prob = model.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        ys = torch.cat([ys,
                        torch.ones(1, 1).type_as(src.data).fill_(next_word)], dim=0)
        if next_word == EOS_IDX:
            break
    return ys

# --- Test the translation ---
def translate(model: torch.nn.Module, src_seq: torch.Tensor):
    model.eval()
    src = src_seq.view(-1, 1)
    num_tokens = src.shape[0]
    src_mask = (torch.zeros(num_tokens, num_tokens)).type(torch.bool)
    tgt_tokens = greedy_decode(
        model,  src, src_mask, max_len=num_tokens + 5, start_symbol=BOS_IDX).flatten()
    return " ".join([str(tok.item()) for tok in tgt_tokens])


# --- Example Usage ---
print("\n--- Inference Example ---")
test_sequence = torch.tensor([BOS_IDX, 7, 2, 9, 1, 5, EOS_IDX])
print(f"Source Sequence: {' '.join([str(i.item()) for i in test_sequence])}")
print(f"Predicted Target: {translate(transformer, test_sequence)}")
print(f"Expected Target: {BOS_IDX} 1 2 5 7 9 {EOS_IDX}")
